# Geodesics and Optimal Transformation Paths

This notebook explores how to compute and interpret geodesic paths on the learned manifold.

## Table of Contents
1. [Geodesics Revisited](#1-geodesics-revisited)
2. [Computing Geodesics via Optimization](#2-computing-geodesics-via-optimization)
3. [Path Length and Curvature](#3-path-length-and-curvature)
4. [Interpreting Geodesic Paths](#4-interpreting-geodesic-paths)
5. [Connection to λ-Scheduling in FEP](#5-connection-to-λ-scheduling)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 1. Geodesics Revisited

### Definition

A **geodesic** is a curve $\gamma(t)$ that minimizes the path length functional:

$$L[\gamma] = \int_0^1 \sqrt{\dot{\gamma}(t)^T \mathbf{M}(\gamma(t)) \dot{\gamma}(t)} \, dt$$

subject to boundary conditions $\gamma(0) = \mathbf{h}_A$ and $\gamma(1) = \mathbf{h}_B$.

### For Constant Metrics

When the metric $\mathbf{M}$ is constant (doesn't depend on position), geodesics are straight lines in the original coordinates. However, their **length** is measured differently:

$$L = \sqrt{(\mathbf{h}_B - \mathbf{h}_A)^T \mathbf{M} (\mathbf{h}_B - \mathbf{h}_A)}$$

### For Position-Dependent Metrics

If $\mathbf{M}(\mathbf{h})$ varies with position, geodesics can be curved. They satisfy the geodesic equation:

$$\ddot{\gamma}^k + \Gamma^k_{ij} \dot{\gamma}^i \dot{\gamma}^j = 0$$

where $\Gamma^k_{ij}$ are the Christoffel symbols derived from the metric.

## 2. Computing Geodesics via Optimization

### Discretized Path Approach

Instead of solving differential equations, we discretize the path into $N$ waypoints and optimize their positions:

$$\gamma = [\mathbf{h}_A, \mathbf{w}_1, \mathbf{w}_2, \ldots, \mathbf{w}_{N-2}, \mathbf{h}_B]$$

The total path length is:

$$L \approx \sum_{i=0}^{N-2} \sqrt{(\mathbf{w}_{i+1} - \mathbf{w}_i)^T \mathbf{M} (\mathbf{w}_{i+1} - \mathbf{w}_i)}$$

We minimize this with respect to the intermediate waypoints $\{\mathbf{w}_i\}$.

In [ ]:
def compute_geodesic_path(h_start, h_end, metric_tensor, n_points=50, n_iterations=300, lr=0.01):
    """
    Compute the geodesic path between two points by optimizing intermediate waypoints.
    
    Args:
        h_start: Starting point embedding (1, d)
        h_end: Ending point embedding (1, d)
        metric_tensor: The metric tensor M (d, d)
        n_points: Number of points along the path
        n_iterations: Optimization iterations
        lr: Learning rate
    
    Returns:
        path: (n_points, d) tensor of waypoints
        geodesic_length: Total path length under the metric
        euclidean_length: Straight-line Euclidean distance
    """
    device = h_start.device
    d = h_start.shape[-1]
    
    # Initialize path as straight line
    t = torch.linspace(0, 1, n_points, device=device).unsqueeze(1)
    h_start_flat = h_start.squeeze(0)
    h_end_flat = h_end.squeeze(0)
    
    initial_path = h_start_flat * (1 - t) + h_end_flat * t
    
    # Make intermediate points learnable (fix endpoints)
    waypoints = nn.Parameter(initial_path[1:-1].clone())
    optimizer = torch.optim.Adam([waypoints], lr=lr)
    
    M = metric_tensor.detach()  # Don't backprop through metric during path optimization
    
    loss_history = []
    
    for iteration in range(n_iterations):
        optimizer.zero_grad()
        
        # Construct full path with fixed endpoints
        full_path = torch.cat([
            h_start_flat.unsqueeze(0),
            waypoints,
            h_end_flat.unsqueeze(0)
        ], dim=0)
        
        # Compute segment vectors
        segments = full_path[1:] - full_path[:-1]  # (n_points-1, d)
        
        # Compute Riemannian length of each segment: sqrt(s^T M s)
        segment_lengths = torch.sqrt(
            torch.sum(segments * (segments @ M), dim=1) + 1e-8
        )
        
        # Total path length
        total_length = segment_lengths.sum()
        
        # Add small regularization to encourage smooth paths
        smoothness = torch.sum((full_path[2:] - 2*full_path[1:-1] + full_path[:-2])**2)
        loss = total_length + 0.001 * smoothness
        
        loss.backward()
        optimizer.step()
        
        loss_history.append(total_length.item())
    
    # Final path and metrics
    with torch.no_grad():
        final_path = torch.cat([
            h_start_flat.unsqueeze(0),
            waypoints,
            h_end_flat.unsqueeze(0)
        ], dim=0)
        
        segments = final_path[1:] - final_path[:-1]
        segment_lengths = torch.sqrt(
            torch.sum(segments * (segments @ M), dim=1) + 1e-8
        )
        geodesic_length = segment_lengths.sum().item()
        
        # Euclidean (straight-line) distance
        euclidean_length = torch.norm(h_end_flat - h_start_flat).item()
    
    return final_path, geodesic_length, euclidean_length, loss_history


# Example with anisotropic metric
dim = 2
h_start = torch.tensor([[0.0, 0.0]])
h_end = torch.tensor([[1.0, 1.0]])

# Create anisotropic metric: moving in x is 4x harder than moving in y
M = torch.tensor([[4.0, 0.0], [0.0, 1.0]])

path, geo_len, euc_len, history = compute_geodesic_path(
    h_start, h_end, M, n_points=30, n_iterations=200
)

print(f"Euclidean distance: {euc_len:.4f}")
print(f"Geodesic length: {geo_len:.4f}")
print(f"Ratio (geodesic/euclidean): {geo_len/euc_len:.4f}")

In [ ]:
# Visualize the geodesic
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Path comparison
path_np = path.detach().numpy()
axes[0].plot(path_np[:, 0], path_np[:, 1], 'b-o', markersize=4, label='Geodesic', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'r--', label='Straight line', linewidth=2)
axes[0].scatter([0, 1], [0, 1], s=150, c=['green', 'red'], zorder=5, edgecolors='black')
axes[0].annotate('Start (A)', (0, 0), xytext=(-0.15, -0.1), fontsize=10)
axes[0].annotate('End (B)', (1, 1), xytext=(1.05, 1.05), fontsize=10)

# Draw metric ellipse
theta = np.linspace(0, 2*np.pi, 100)
ellipse_x = 0.3 * np.cos(theta) / 2  # x scaled by sqrt(4) = 2
ellipse_y = 0.3 * np.sin(theta)
axes[0].plot(ellipse_x + 0.5, ellipse_y + 0.5, 'g--', alpha=0.5, label='Metric unit circle')

axes[0].set_xlabel('Dimension 1 (high metric weight)', fontsize=11)
axes[0].set_ylabel('Dimension 2 (low metric weight)', fontsize=11)
axes[0].set_title('Geodesic vs Straight Line\n(Moving horizontally costs more)', fontsize=12)
axes[0].legend()
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Right: Optimization convergence
axes[1].plot(history, 'b-', linewidth=2)
axes[1].set_xlabel('Iteration', fontsize=11)
axes[1].set_ylabel('Path Length', fontsize=11)
axes[1].set_title('Geodesic Optimization Convergence', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNote: The geodesic is still a straight line because our metric is constant.")
print("But the LENGTH is different from Euclidean due to the anisotropic metric.")

## 3. Path Length and Curvature

### Measuring Path Curvature

The **curvature** of a path tells us how much it deviates from a straight line. For geodesics under a constant metric, this is zero. But we can define a "relative curvature" that measures how much longer the geodesic is compared to the Euclidean straight line:

$$\text{Relative Curvature} = \frac{L_{\text{geodesic}}}{L_{\text{euclidean}}} - 1$$

- **0%**: Geodesic equals Euclidean (isotropic metric)
- **Positive**: Geodesic is longer (metric stretches distances)
- **Large positive**: Transformation is much harder than Euclidean distance suggests

### Energy Along the Path

We can also compute the "energy" (metric-weighted distance from start) at each point along the geodesic:

In [ ]:
def compute_path_energy_profile(path, metric_tensor):
    """
    Compute cumulative energy along a path.
    
    Args:
        path: (n_points, d) tensor of waypoints
        metric_tensor: (d, d) metric tensor
    
    Returns:
        energy_profile: (n_points,) cumulative energy at each point
        segment_energies: (n_points-1,) energy of each segment
    """
    M = metric_tensor
    segments = path[1:] - path[:-1]
    
    # Riemannian length of each segment
    segment_energies = torch.sqrt(
        torch.sum(segments * (segments @ M), dim=1) + 1e-8
    )
    
    # Cumulative energy
    energy_profile = torch.zeros(len(path))
    energy_profile[1:] = torch.cumsum(segment_energies, dim=0)
    
    return energy_profile, segment_energies


# Compute energy profile for our geodesic
energy_profile, segment_energies = compute_path_energy_profile(path, M)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Cumulative energy
t_param = np.linspace(0, 1, len(path))
axes[0].plot(t_param, energy_profile.numpy(), 'b-', linewidth=2, label='Riemannian')
axes[0].plot([0, 1], [0, euc_len], 'r--', linewidth=2, label='Euclidean')
axes[0].set_xlabel('Path Parameter t (0 = A, 1 = B)', fontsize=11)
axes[0].set_ylabel('Cumulative Distance', fontsize=11)
axes[0].set_title('Energy Profile Along Path', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: Segment energies
t_segments = np.linspace(0, 1, len(segment_energies))
axes[1].bar(t_segments, segment_energies.numpy(), width=0.03, alpha=0.7)
axes[1].axhline(y=segment_energies.mean().item(), color='r', linestyle='--', label='Mean')
axes[1].set_xlabel('Path Parameter t', fontsize=11)
axes[1].set_ylabel('Segment Energy', fontsize=11)
axes[1].set_title('Energy per Segment', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

relative_curvature = (geo_len / euc_len - 1) * 100
print(f"\nRelative curvature: {relative_curvature:.1f}%")
print(f"Interpretation: The transformation is {relative_curvature:.1f}% harder than Euclidean distance suggests.")

## 4. Interpreting Geodesic Paths

### What the Geodesic Tells Us

The geodesic path has several important interpretations:

1. **Optimal Transformation Route**: The path that minimizes total "difficulty" from A to B

2. **Intermediate States**: Points along the geodesic represent intermediate molecular states

3. **Bottlenecks**: Regions where segment energy is high indicate difficult transitions

### Visualizing in 3D

Let's create a more complex example with a position-dependent metric that creates barriers:

In [ ]:
def create_potential_energy_surface(grid_size=50):
    """
    Create a double-well potential energy surface for visualization.
    
    Returns:
        X, Y: Meshgrid coordinates
        E: Energy values at each point
    """
    x = np.linspace(-1.5, 1.5, grid_size)
    y = np.linspace(-1.0, 1.0, grid_size)
    X, Y = np.meshgrid(x, y)
    
    # Double-well potential
    # Wells at (-1, 0) and (1, 0)
    well_1 = np.exp(-((X + 1)**2 + Y**2) / 0.2)
    well_2 = np.exp(-((X - 1)**2 + Y**2) / 0.2)
    
    # Barrier at origin
    barrier = 0.8 * np.exp(-(X**2 + Y**2) / 0.3)
    
    # Combined energy
    E = 0.3 * (X**2 + 0.5*Y**2) - 1.2*well_1 - 1.2*well_2 + barrier
    E = E - E.min()  # Shift minimum to zero
    
    return X, Y, E


def compute_minimum_energy_path(E, X, Y, start_idx, end_idx, n_iterations=500):
    """
    Compute minimum energy path (MEP) using the nudged elastic band method.
    
    This is a simplified version for visualization.
    """
    from scipy.interpolate import RegularGridInterpolator
    
    # Create interpolator for energy
    interp = RegularGridInterpolator(
        (X[0, :], Y[:, 0]), E.T, method='linear'
    )
    
    # Start and end points
    start = np.array([X[start_idx], Y[start_idx]])
    end = np.array([X[end_idx], Y[end_idx]])
    
    # Initialize path as straight line
    n_points = 30
    path = np.linspace(start, end, n_points)
    
    # Simple gradient descent on the energy surface
    lr = 0.01
    for _ in range(n_iterations):
        # Compute energy gradient at each point (finite differences)
        delta = 0.01
        for i in range(1, n_points - 1):
            pt = path[i]
            
            # Gradient
            grad = np.zeros(2)
            for d in range(2):
                pt_plus = pt.copy()
                pt_plus[d] += delta
                pt_minus = pt.copy()
                pt_minus[d] -= delta
                
                # Clamp to grid bounds
                pt_plus = np.clip(pt_plus, [X.min(), Y.min()], [X.max(), Y.max()])
                pt_minus = np.clip(pt_minus, [X.min(), Y.min()], [X.max(), Y.max()])
                
                grad[d] = (interp(pt_plus) - interp(pt_minus)) / (2 * delta)
            
            # Project gradient perpendicular to path direction
            tangent = path[i+1] - path[i-1]
            tangent = tangent / (np.linalg.norm(tangent) + 1e-8)
            perp_grad = grad - np.dot(grad, tangent) * tangent
            
            # Update
            path[i] -= lr * perp_grad
            path[i] = np.clip(path[i], [X.min() + 0.1, Y.min() + 0.1], [X.max() - 0.1, Y.max() - 0.1])
    
    return path


# Create energy surface and MEP
X, Y, E = create_potential_energy_surface(50)

# Manually define start/end indices (wells at x=-1 and x=1)
start = np.array([-1.0, 0.0])
end = np.array([1.0, 0.0])

# Create straight line path for comparison
n_path_points = 30
straight_path = np.linspace(start, end, n_path_points)

# Create MEP (minimum energy path)
mep = compute_minimum_energy_path(E, X, Y, (25, 8), (25, 42), n_iterations=300)

print("Energy surface and paths computed.")

In [ ]:
# Visualize the energy landscape with paths
fig = plt.figure(figsize=(16, 6))

# 3D Surface
ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(X, Y, E, cmap='viridis', alpha=0.8, edgecolor='none')
ax1.set_xlabel('X (Reaction Coord 1)')
ax1.set_ylabel('Y (Reaction Coord 2)')
ax1.set_zlabel('Energy')
ax1.set_title('Free Energy Surface')
ax1.view_init(elev=30, azim=45)

# Contour with paths
ax2 = fig.add_subplot(132)
cs = ax2.contour(X, Y, E, levels=15, cmap='viridis')
ax2.clabel(cs, inline=True, fontsize=8)
ax2.plot(straight_path[:, 0], straight_path[:, 1], 'r--', linewidth=2, label='Straight line')
ax2.plot(mep[:, 0], mep[:, 1], 'b-', linewidth=3, label='Min. Energy Path')
ax2.scatter([-1, 1], [0, 0], s=100, c=['green', 'red'], zorder=5, edgecolors='black')
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('Contour Map with Paths')
ax2.legend()

# Energy profile along paths
ax3 = fig.add_subplot(133)
from scipy.interpolate import RegularGridInterpolator
interp = RegularGridInterpolator((X[0, :], Y[:, 0]), E.T, method='linear')

t = np.linspace(0, 1, len(straight_path))
straight_energy = [interp(pt) for pt in straight_path]
mep_energy = [interp(pt) for pt in mep]

ax3.plot(t, straight_energy, 'r--', linewidth=2, label='Straight line')
ax3.plot(t, mep_energy, 'b-', linewidth=2, label='Min. Energy Path')
ax3.fill_between(t, straight_energy, alpha=0.3, color='red')
ax3.set_xlabel('Path Parameter (0→1)')
ax3.set_ylabel('Energy')
ax3.set_title('Energy Along Paths')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("- The straight line goes directly through the barrier (high energy)")
print("- The MEP curves to avoid the barrier, going through the saddle point")
print("- The geodesic under our learned metric approximates the MEP")

## 5. Connection to λ-Scheduling in FEP <a name="5-connection-to-λ-scheduling"></a>

### The λ Parameter in FEP

In Free Energy Perturbation, the transformation from molecule A to B is controlled by a coupling parameter $\lambda \in [0, 1]$:

$$H(\lambda) = (1 - \lambda) H_A + \lambda H_B$$

- $\lambda = 0$: System is fully molecule A
- $\lambda = 1$: System is fully molecule B
- $\lambda \in (0, 1)$: Hybrid "alchemical" state

### Optimal λ-Spacing

The key insight is that **uniform λ-spacing is suboptimal**. Some regions of $\lambda$ have rapidly changing free energy (hard to sample), while others are smooth (easy).

### Connection to Geodesics

The geodesic path naturally suggests an optimal λ-schedule:

$$\lambda_i^{\text{optimal}} = \frac{\text{cumulative geodesic length up to point } i}{\text{total geodesic length}}$$

This spreads the sampling effort according to transformation difficulty.

In [ ]:
def compute_optimal_lambda_schedule(path, metric_tensor, n_windows=12):
    """
    Compute optimal λ-window spacing based on geodesic path.
    
    Args:
        path: (n_points, d) geodesic path
        metric_tensor: (d, d) metric
        n_windows: Number of λ-windows desired
    
    Returns:
        lambda_uniform: Uniform λ spacing
        lambda_optimal: Optimal λ spacing based on geodesic
    """
    # Compute cumulative path length
    energy_profile, _ = compute_path_energy_profile(path, metric_tensor)
    cumulative = energy_profile.numpy()
    total_length = cumulative[-1]
    
    # Normalize to [0, 1]
    cumulative_normalized = cumulative / total_length
    
    # Uniform spacing in path parameter
    uniform_indices = np.linspace(0, len(path)-1, n_windows).astype(int)
    lambda_uniform = np.linspace(0, 1, n_windows)
    
    # Optimal: uniform spacing in cumulative distance
    target_distances = np.linspace(0, 1, n_windows)
    lambda_optimal = np.interp(target_distances, cumulative_normalized, np.linspace(0, 1, len(path)))
    
    return lambda_uniform, lambda_optimal, cumulative_normalized


# Compute optimal λ-schedule for our geodesic
lambda_uniform, lambda_optimal, cumulative_norm = compute_optimal_lambda_schedule(
    path, M, n_windows=12
)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: λ-schedules comparison
ax1 = axes[0]
ax1.scatter(range(len(lambda_uniform)), lambda_uniform, s=100, c='red', 
            marker='o', label='Uniform λ', zorder=5)
ax1.scatter(range(len(lambda_optimal)), lambda_optimal, s=100, c='blue', 
            marker='s', label='Optimal λ', zorder=5)
ax1.plot(range(len(lambda_uniform)), lambda_uniform, 'r--', alpha=0.5)
ax1.plot(range(len(lambda_optimal)), lambda_optimal, 'b-', alpha=0.5)
ax1.set_xlabel('Window Index', fontsize=11)
ax1.set_ylabel('λ Value', fontsize=11)
ax1.set_title('Uniform vs Optimal λ-Spacing', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Cumulative distance showing where windows should be placed
ax2 = axes[1]
t = np.linspace(0, 1, len(cumulative_norm))
ax2.plot(t, cumulative_norm, 'b-', linewidth=2, label='Cumulative distance')
ax2.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Linear (uniform)')

# Mark optimal window positions
for lam in lambda_optimal:
    ax2.axvline(x=lam, color='blue', alpha=0.3, linestyle=':')

ax2.set_xlabel('Path Parameter t', fontsize=11)
ax2.set_ylabel('Cumulative Riemannian Distance (normalized)', fontsize=11)
ax2.set_title('Optimal Window Placement', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nUniform λ-spacing:", np.round(lambda_uniform, 3))
print("Optimal λ-spacing:", np.round(lambda_optimal, 3))
print("\nNote: Optimal spacing concentrates windows in high-difficulty regions.")

## Summary

### Key Takeaways

1. **Geodesics** are shortest paths under the learned Riemannian metric

2. **Computing geodesics** via waypoint optimization is efficient and differentiable

3. **Relative curvature** measures how much harder a transformation is than Euclidean distance suggests

4. **Energy profile** along the geodesic shows where bottlenecks occur

5. **Optimal λ-scheduling** can be derived from the geodesic to improve FEP efficiency

### Practical Applications

| Insight from Geodesic | Practical Action |
|----------------------|------------------|
| High relative curvature | Allocate more sampling time |
| Energy spike in middle | Add intermediate λ-windows there |
| Nearly straight path | Standard uniform λ-spacing is fine |

### Next Steps

In the next notebook, we'll explore:
- Visualizing the full energy landscape
- 3D manifold representations
- Interactive exploration tools